# 04 · Eleição do vencedor e promoção controlada

**Etapa 1 — vencedor do ciclo:** maior média de AUC-PR na validação cruzada (notebook 03). Empate: menor desvio-padrão.

**Etapa 2 — vencedor × vigente:** os dois são avaliados na **mesma validação do ciclo** (dados que nenhum dos dois viu no treino).

| Situação | Decisão |
|---|---|
| Candidato não supera o modelo aleatório (AUC-PR ≤ prevalência) | REJEITADO, vigente mantido |
| Não existe vigente | PROMOVIDO |
| Candidato ≥ vigente + 0,005 | PROMOVIDO |
| Caso contrário | MANTIDO |

O teste (15% reservado) só é usado no fim, para reportar o desempenho do modelo que ficou vigente. Qualquer erro neste notebook interrompe o workflow **antes** de o novo estado ser salvo, então nunca há promoção indevida.

In [1]:
import os, sys, warnings
from pathlib import Path
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore", category=UserWarning)
# Localiza a raiz do repositório e torna o pacote src importável
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
os.chdir(RAIZ); sys.path.insert(0, str(RAIZ))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from src.utils import carregar_config, configurar_mlflow, ler_contexto, atualizar_contexto, tags_ciclo, caminho, salvar_json
cfg = carregar_config()
pd.set_option("display.width", 160)

In [2]:
import joblib
from src.preprocessamento import separar_xy
from src.modelagem import calcular_metricas
from src.promocao import eleger_vencedor, decidir_promocao, carregar_vigente, promover, registrar_decisao, ler_manifesto
from src.utils import resumo_github

ctx = ler_contexto(cfg)
metrica, direcao = cfg["avaliacao"]["metrica_principal"], cfg["avaliacao"]["direcao"]
pct = cfg["negocio"]["percentual_contato"]
proc = caminho(cfg["dados"]["processed_dir"])
validacao = pd.read_csv(proc / "validacao.csv"); teste = pd.read_csv(proc / "teste.csv")
X_va, y_va = separar_xy(validacao, cfg); X_te, y_te = separar_xy(teste, cfg)
cv = pd.read_csv(caminho(cfg["dados"]["state_dir"]) / "cv_ciclo.csv")
print(ctx["cycle_id"], ctx["data_version"])

c002-20261117-1951 v002-db4f8bd6


## 1. Vencedor do ciclo (A × B)

In [3]:
vencedor, ranking = eleger_vencedor(cv, metrica, direcao)
display(ranking.round(4))
cand = ctx["candidatos"][vencedor]
pipe_cand = joblib.load(caminho(cfg["dados"]["state_dir"]) / "candidatos" / f"{vencedor}.joblib")
print(f"Vencedor do ciclo: Modelo {vencedor} ({cand['algoritmo']})")

,modelo,algoritmo,average_precision_media,average_precision_desvio
0,B,random_forest,0.2058,0.0229
1,A,regressao_logistica,0.1917,0.0270


Vencedor do ciclo: Modelo B (random_forest)


## 2. Vencedor × modelo vigente (mesma validação)

In [4]:
vigente, manif_ant = carregar_vigente()   # vem do armazenamento persistente (repositório), não do runner
m_cand = calcular_metricas(y_va, pipe_cand.predict_proba(X_va)[:, 1], pct)
m_vig = calcular_metricas(y_va, vigente.predict_proba(X_va)[:, 1], pct) if vigente is not None else None

# Verificações de sanidade: falhas aqui interrompem o fluxo antes da promoção
assert len(ctx["features_eleitas"]) > 0, "Conjunto de features vazio"
assert np.isfinite(m_cand[metrica]), "Métrica do candidato inválida"
assert list(pipe_cand.named_steps["selecao"].features) == ctx["features_eleitas"], "Candidato não usa as features eleitas do ciclo"

piso = m_cand["prevalencia"]
decisao = decidir_promocao(m_cand[metrica], None if m_vig is None else m_vig[metrica], piso, cfg)

comp = pd.DataFrame({f"candidato ({cand['algoritmo']}, {ctx['cycle_id']})": m_cand,
                     **({f"vigente ({manif_ant['algoritmo']}, {manif_ant['cycle_id']})": m_vig} if m_vig else {})}).round(4)
display(comp)
display(Markdown(f"### Decisão: **{decisao['decisao']}**  \n{decisao['motivo']}"))

,"candidato (random_forest, c002-20261117-1951)","vigente (random_forest, c001-20261116-2102)"
average_precision,0.2043,0.1067
roc_auc,0.7036,0.6097
f1,0.2013,0.1380
precision,0.1272,0.0823
recall,0.4825,0.4280
prevalencia,0.0575,0.0575
conversao_top,0.1355,0.0873
lift_top,2.3557,1.5185
captura_top,0.4708,0.3035


### Decisão: **PROMOVIDO**  
Candidato (0.2043) supera o vigente (0.1067) em pelo menos 0.005 na validação do ciclo.

## 3. Aplicação da decisão e manifesto

In [5]:
if decisao["decisao"] == "PROMOVIDO":
    manifesto = promover(pipe_cand, {
        "cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "modelo": vencedor,
        "algoritmo": cand["algoritmo"], "run_id": cand["run_id"], "features": ctx["features_eleitas"],
        "metrica_principal": metrica, "valor_validacao": m_cand[metrica],
        "cv_media": cand[f"cv_{metrica}_media"], "decisao": decisao,
        "drift_alerta_no_ciclo": ctx.get("drift_alerta"), "features_com_drift": ctx.get("features_com_drift"),
        "gatilho": ctx["execucao"]["gatilho"]})
    vigente = pipe_cand
else:
    manifesto = ler_manifesto()
    if manifesto is None:
        raise RuntimeError("Nenhum modelo vigente e o candidato foi rejeitado: pipeline interrompido.")
print(json.dumps(manifesto, indent=2, ensure_ascii=False))

{
  "cycle_id": "c002-20261117-1951",
  "data_version": "v002-db4f8bd6",
  "modelo": "B",
  "algoritmo": "random_forest",
  "run_id": "95017d4674bf4200a7adcbc8d2fa85ce",
  "features": [
    "euribor3m",
    "month",
    "emp.var.rate",
    "nr.employed",
    "cons.conf.idx",
    "cons.price.idx",
    "job",
    "age"
  ],
  "metrica_principal": "average_precision",
  "valor_validacao": 0.2043096110095641,
  "cv_media": 0.20579091226513682,
  "decisao": {
    "decisao": "PROMOVIDO",
    "motivo": "Candidato (0.2043) supera o vigente (0.1067) em pelo menos 0.005 na validação do ciclo."
  },
  "drift_alerta_no_ciclo": true,
  "features_com_drift": [
    "month",
    "emp.var.rate",
    "cons.price.idx",
    "euribor3m",
    "nr.employed",
    "cons.conf.idx",
    "pdays",
    "contact",
    "poutcome"
  ],
  "gatilho": "schedule",
  "arquivo_modelo": "models/historico/c002-20261117-1951_random_forest.joblib",
  "promovido_em": "2026-11-17T19:53:14+00:00",
  "substituiu": "c001-20261116-21

## 4. Avaliação final do modelo vigente no teste reservado

In [6]:
m_teste = calcular_metricas(y_te, vigente.predict_proba(X_te)[:, 1], pct)
display(pd.Series(m_teste, name="teste").round(4).to_frame())

mlflow = configurar_mlflow(cfg)
with mlflow.start_run(run_name=f"{ctx['cycle_id']}_04_decisao", tags={**tags_ciclo(ctx, "04_promocao"), "decisao": decisao["decisao"]}) as run:
    mlflow.log_params({"vencedor_ciclo": vencedor, "algoritmo_vencedor": cand["algoritmo"], "run_id_vencedor": cand["run_id"],
                       "decisao": decisao["decisao"], "motivo": decisao["motivo"][:250],
                       "limiar_melhoria": cfg["promocao"]["limiar_melhoria"],
                       "vigente_cycle_id": manifesto["cycle_id"], "vigente_run_id": manifesto["run_id"]})
    mlflow.log_metrics({f"candidato_val_{metrica}": m_cand[metrica], f"piso_aleatorio_{metrica}": piso,
                        **({f"vigente_anterior_val_{metrica}": m_vig[metrica]} if m_vig else {}),
                        **{f"vigente_teste_{k}": v for k, v in m_teste.items()}})
    mlflow.log_dict(manifesto, "modelo_vigente.json")
    mlflow.log_table(ranking, "ranking_vencedor.json")

linha = {"cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "gatilho": ctx["execucao"]["gatilho"],
         "lote": ctx["lote"].get("lote"), "lote_novo": ctx["lote"]["novo"], "linhas_lote": ctx["lote"]["linhas"],
         "features_eleitas": "|".join(ctx["features_eleitas"]), "n_features": len(ctx["features_eleitas"]),
         "cv_A": ctx["candidatos"]["A"][f"cv_{metrica}_media"], "cv_B": ctx["candidatos"]["B"][f"cv_{metrica}_media"],
         "vencedor": f"{vencedor}-{cand['algoritmo']}", f"candidato_val_{metrica}": m_cand[metrica],
         f"vigente_anterior_val_{metrica}": None if m_vig is None else m_vig[metrica],
         "decisao": decisao["decisao"], "vigente_apos": f"{manifesto['cycle_id']}-{manifesto['algoritmo']}",
         f"vigente_teste_{metrica}": m_teste[metrica], "vigente_teste_lift_top": m_teste["lift_top"],
         "drift_alerta": ctx.get("drift_alerta"), "psi_max": ctx.get("psi_max"), "run_decisao": run.info.run_id}
historico = registrar_decisao(linha)
display(historico.tail(5))

resumo_github(f"### Decisão: **{decisao['decisao']}**\n\n{decisao['motivo']}\n\n"
              f"| | valor |\n|---|---|\n| vencedor do ciclo | {vencedor} ({cand['algoritmo']}) |\n"
              f"| features | {', '.join(ctx['features_eleitas'])} |\n| vigente | {manifesto['cycle_id']} ({manifesto['algoritmo']}) |\n"
              f"| {metrica} no teste (vigente) | {m_teste[metrica]:.4f} |\n")
atualizar_contexto(cfg, decisao=decisao, run_decisao=run.info.run_id)

,teste
average_precision,0.1831
roc_auc,0.6553
f1,0.2040
precision,0.1349
recall,0.4184
prevalencia,0.0659
conversao_top,0.1380
lift_top,2.0932
captura_top,0.4184


,cycle_id,data_version,gatilho,lote,lote_novo,linhas_lote,features_eleitas,n_features,cv_A,cv_B,vencedor,candidato_val_average_precision,vigente_anterior_val_average_precision,decisao,vigente_apos,vigente_teste_average_precision,vigente_teste_lift_top,drift_alerta,psi_max,run_decisao
0,c001-20261116-2102,v001-44df464c,schedule,0,True,27680,euribor3m|month|education|job|age|emp.var.rate...,8,0.090226,0.100393,B-random_forest,0.104494,NaN,PROMOVIDO,c001-20261116-2102-random_forest,0.093592,1.390251,False,NaN,1456d4163be643ec93414675f1013b68
1,c002-20261117-1951,v002-db4f8bd6,schedule,1,True,2750,euribor3m|month|emp.var.rate|nr.employed|cons....,8,0.191662,0.205791,B-random_forest,0.204310,0.106718,PROMOVIDO,c002-20261117-1951-random_forest,0.183092,2.093245,True,16.475,559ce72fad6c46a6862a5f5f2141eb78


{'cycle_id': 'c002-20261117-1951',
 'data_version': 'v002-db4f8bd6',
 'execucao': {'gatilho': 'schedule',
  'github_run_id': '44162934462',
  'github_run_url': 'https://github.com/weillerjulia/n3-mlops-bank-marketing/actions/runs/44162934462',
  'commit': '0b15879',
  'iniciado_em': '2026-11-17T19:51:41+00:00'},
 'lote': {'novo': True,
  'lote': 1,
  'arquivo': 'data/raw/lote_001.csv',
  'periodos': ['2008-12', '2009-03', '2009-04'],
  'linhas': 2750,
  'sha256': '19adc497396f',
  'ingerido_em': '2026-11-17T19:51:41+00:00'},
 'drift_alerta': True,
 'features_com_drift': ['month',
  'emp.var.rate',
  'cons.price.idx',
  'euribor3m',
  'nr.employed',
  'cons.conf.idx',
  'pdays',
  'contact',
  'poutcome'],
 'psi_max': 16.475,
 'run_etl': '9461e2270d314249a2c96b04db8544d4',
 'features_eleitas': ['euribor3m',
  'month',
  'emp.var.rate',
  'nr.employed',
  'cons.conf.idx',
  'cons.price.idx',
  'job',
  'age'],
 'run_votacao': '70adf77f35144b10937f39a68553198b',
 'candidatos': {'A': {'run